# Session 5 — Airflow Pipelines

In this session we build and run an **end-to-end ML pipeline as an Airflow DAG**
using the iris dataset. By the end you will:

- Understand the Airflow TaskFlow API (`@dag`, `@task`, XCom)
- Write a real DAG that loads, validates, featurizes, trains, and evaluates a model
- Use `airflow dags test` to run it locally without a running scheduler
- See what the Airflow UI would show (task states, logs, run history)

---

**Key concept recap from the materials:**

```
load_data ──▶ validate_data ──▶ featurize ──▶ train_model ──▶ evaluate
```

Each arrow is a **task dependency**. The scheduler will not start a task until all
upstream tasks have succeeded.

## Part 1 — Setup

Airflow needs two things before you can run anything:

1. `AIRFLOW_HOME` — the directory where Airflow stores its config, SQLite DB, DAG files, and logs.
2. An initialised **metadata database** — run once to create all the tables Airflow needs.

We point `AIRFLOW_HOME` at a local subfolder so everything stays self-contained.

In [ ]:
import subprocess, sys, os, pathlib, io

# ── Paths (all relative — no hardcoded absolute paths) ────────────────────────
# __file__ is not defined in notebooks; instead we locate Session_5 by searching
# upward from the kernel's cwd, which Jupyter/VS Code sets to the notebook's
# own directory when you open it normally.
SESSION_DIR = pathlib.Path.cwd()
if SESSION_DIR.name != "Session_5":
    for p in [SESSION_DIR] + list(SESSION_DIR.parents):
        if p.name == "Session_5":
            SESSION_DIR = p
            break

AIRFLOW_HOME = SESSION_DIR / "airflow_home"
DAGS_DIR     = AIRFLOW_HOME / "dags"
DAGS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = (SESSION_DIR.parent / "Session_1" / "iris.csv").resolve()

# ── Python used by this kernel (so subprocess picks up the right packages) ────
PYTHON = sys.executable

# ── Helper: run an Airflow CLI command ────────────────────────────────────────
def airflow(*args, capture=False):
    env = {**os.environ, "AIRFLOW_HOME": str(AIRFLOW_HOME)}
    cmd = [PYTHON, "-m", "airflow", *args]
    result = subprocess.run(cmd, env=env, capture_output=capture, text=True)
    if capture:
        return result.stdout + result.stderr
    return result.returncode

print(f"SESSION_DIR  : {SESSION_DIR}")
print(f"AIRFLOW_HOME : {AIRFLOW_HOME}")
print(f"DATA_PATH    : {DATA_PATH} (exists={DATA_PATH.exists()})")
print(f"Python       : {PYTHON}")

In [ ]:
# Install Apache Airflow if not already present
# The constraints file pins every transitive dependency so versions don't clash.
try:
    import airflow
    print(f"Apache Airflow {airflow.__version__} already installed — skipping.")
except ImportError:
    print("Installing Apache Airflow 2.9.2 (this may take ~2 minutes)...")
    constraints = (
        "https://raw.githubusercontent.com/apache/airflow/"
        "constraints-2.9.2/constraints-3.12.txt"
    )
    subprocess.run(
        [PYTHON, "-m", "pip", "install", "apache-airflow==2.9.2",
         "--constraint", constraints, "-q"],
        check=True,
    )
    import airflow
    print(f"Installed Apache Airflow {airflow.__version__}")

In [ ]:
# Initialise the metadata database (SQLite by default — fine for local dev)
# This is idempotent: safe to re-run if the DB already exists.
import airflow
print(f"Airflow version : {airflow.__version__}")
print(f"Initialising DB at {AIRFLOW_HOME / 'airflow.db'} ...")

env = {**os.environ, "AIRFLOW_HOME": str(AIRFLOW_HOME)}
result = subprocess.run(
    [PYTHON, "-m", "airflow", "db", "init"],
    env=env, capture_output=True, text=True
)
# Print only the last few lines — the full migration log is very verbose
last_lines = (result.stdout + result.stderr).strip().splitlines()
for line in last_lines[-6:]:
    print(line)

## Part 2 — Airflow core concepts

### DAG
A DAG file is a plain Python file placed in `$AIRFLOW_HOME/dags/`. The scheduler
scans that directory on a configurable interval (default: 30 s). When it finds a
new file it **imports** it — so the file must be importable (no side effects at
module level except DAG registration).

### `@task` and XCom
The **TaskFlow API** (Airflow 2.0+) lets you write tasks as plain Python functions
decorated with `@task`. The return value is automatically serialised as an **XCom**
("cross-communication") entry in the metadata DB and passed as an argument to
downstream tasks.

```python
@task
def load_data() -> str:
    return df.to_json(orient="records")   # stored in XCom

@task
def validate(raw_json: str):             # receives XCom value as argument
    ...
```

> **XCom size limit** — the default SQLite / Postgres XCom backend has a ~48 KB
> limit. Pass **paths to files** (on S3, GCS, or a shared NFS mount) for large
> data; only pass metadata (row counts, accuracy, file paths) as XCom values.

### Task dependencies
Calling a `@task` function returns an `XComArg` (a placeholder, not the actual
value). When you pass one as an argument to another `@task`, Airflow infers the
dependency.

```python
raw   = load_data()       # load_data runs first
clean = validate(raw)     # validate runs after load_data
train(clean)              # train runs after validate
```

You can also express dependencies explicitly with `>>` for non-TaskFlow operators:

```python
load_task >> validate_task >> train_task
```

### `schedule` and `catchup`
- **`schedule=None`** — manual trigger only (what we use in this demo).
- **`schedule="0 6 * * *"`** — cron string: run at 06:00 every day.
- **`catchup=False`** — don't backfill runs that should have happened between
  `start_date` and now. Almost always set this to `False` for ML pipelines.

## Part 3 — Writing the DAG

We write the DAG file to disk programmatically so the notebook is self-contained.
In a real project the DAG file would just live in your git repository under
`dags/` and be deployed to `AIRFLOW_HOME/dags/` by your CI/CD pipeline.

In [ ]:
dag_code = '''
from __future__ import annotations

import io
import pathlib
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from airflow.decorators import dag, task

# Resolve the iris.csv path relative to this DAG file so it works
# regardless of the current working directory when Airflow executes tasks.
DATA_PATH = str(
    pathlib.Path(__file__).resolve().parent.parent.parent.parent
    / "Session_1" / "iris.csv"
)


@dag(
    dag_id="iris_pipeline",
    start_date=datetime(2024, 1, 1),
    schedule=None,       # manual trigger only
    catchup=False,
    tags=["iris", "demo"],
    doc_md="""
    ## Iris ML Pipeline

    End-to-end pipeline for the iris dataset:
    `load_data → validate_data → featurize → train_model → evaluate`
    """,
)
def iris_pipeline():

    @task
    def load_data() -> str:
        """Read iris.csv and return it as a JSON string (passed via XCom)."""
        df = pd.read_csv(DATA_PATH)
        print(f"Loaded {len(df)} rows | columns: {df.columns.tolist()}")
        return df.to_json(orient="records")

    @task
    def validate_data(raw_json: str) -> str:
        """Assert schema, nulls, and numeric ranges. Raises on failure."""
        df = pd.read_json(io.StringIO(raw_json), orient="records")

        expected_cols = {"septal_length", "sepal_width",
                         "petal_length", "petal_width", "class"}
        missing = expected_cols - set(df.columns)
        assert not missing, f"Missing columns: {missing}"
        assert df.isnull().sum().sum() == 0, "Null values detected"
        assert df["septal_length"].between(0, 20).all(), \
            "septal_length contains out-of-range values"
        assert df["sepal_width"].between(0, 20).all(), \
            "sepal_width contains out-of-range values"

        print(
            f"Validation passed | {len(df)} rows | "
            f"{df[\'class\'].nunique()} classes: {df[\'class\'].unique().tolist()}"
        )
        return raw_json

    @task
    def featurize(raw_json: str) -> dict:
        """Encode the class label and split into features / targets."""
        df = pd.read_json(io.StringIO(raw_json), orient="records")
        le = LabelEncoder()
        X = df.drop(columns=["class"]).values.tolist()
        y = le.fit_transform(df["class"]).tolist()
        print(
            f"Feature matrix: ({len(X)} rows x {len(X[0])} cols) | "
            f"Classes: {le.classes_.tolist()}"
        )
        return {"X": X, "y": y, "classes": le.classes_.tolist()}

    @task
    def train_model(features: dict) -> dict:
        """Train a LogisticRegression and return predictions for evaluation."""
        X = np.array(features["X"])
        y = np.array(features["y"])
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        model = LogisticRegression(max_iter=300, random_state=42)
        model.fit(X_train, y_train)
        preds = model.predict(X_test).tolist()
        print(
            f"Trained on {len(X_train)} samples | "
            f"Test set size: {len(X_test)}"
        )
        return {
            "y_test": y_test.tolist(),
            "preds": preds,
            "classes": features["classes"],
        }

    @task
    def evaluate(results: dict) -> float:
        """Compute and print accuracy + classification report."""
        acc = accuracy_score(results["y_test"], results["preds"])
        report = classification_report(
            results["y_test"],
            results["preds"],
            target_names=results["classes"],
        )
        print(f"\n{\"=\" * 50}")
        print(f"  Accuracy : {acc:.4f}")
        print(f"{\"=\" * 50}")
        print(report)
        return float(acc)

    # ── Wire tasks together (defines the DAG edges) ────────────────────────────
    raw    = load_data()
    valid  = validate_data(raw)
    feats  = featurize(valid)
    result = train_model(feats)
    evaluate(result)


iris_pipeline()   # instantiate — Airflow scans for this at module level
'''

dag_path = DAGS_DIR / "iris_pipeline.py"
dag_path.write_text(dag_code)
print(f"DAG written to: {dag_path}")
print(f"File size     : {dag_path.stat().st_size} bytes")

## Part 4 — Exploring the DAG with the CLI

The Airflow CLI is the fastest way to inspect and test DAGs without starting
the full webserver + scheduler stack.

In [ ]:
# List all discovered DAGs
# The scheduler does this same scan to decide what to schedule.
print("=== airflow dags list ===")
env = {**os.environ, "AIRFLOW_HOME": str(AIRFLOW_HOME)}
result = subprocess.run(
    [PYTHON, "-m", "airflow", "dags", "list"],
    env=env, capture_output=True, text=True
)
output = result.stdout + result.stderr
# Filter noise — keep only header/data lines
for line in output.splitlines():
    if any(kw in line for kw in ["dag_id", "iris", "==", "─", "|"]):
        print(line)

In [ ]:
# List all tasks in the iris_pipeline DAG
# This shows the task_id of each node in the graph.
print("=== airflow tasks list iris_pipeline ===")
result = subprocess.run(
    [PYTHON, "-m", "airflow", "tasks", "list", "iris_pipeline"],
    env=env, capture_output=True, text=True
)
output = (result.stdout + result.stderr).strip()
for line in output.splitlines():
    if line.strip() and "WARNING" not in line and "UserWarning" not in line:
        print(line)

In [ ]:
# Show the dependency tree
# --tree renders a text-based graph of the DAG structure.
print("=== airflow tasks list iris_pipeline --tree ===")
result = subprocess.run(
    [PYTHON, "-m", "airflow", "tasks", "list", "iris_pipeline", "--tree"],
    env=env, capture_output=True, text=True
)
output = result.stdout + result.stderr
for line in output.splitlines():
    if "WARNING" not in line and "UserWarning" not in line and "graphviz" not in line:
        print(line)

## Part 5 — Running the pipeline

`airflow dags test <dag_id> <execution_date>` runs all tasks in the DAG
**synchronously in the current process** — no scheduler or worker needed.

This is the standard way to develop and debug Airflow DAGs locally. The
`execution_date` is required but arbitrary when `schedule=None`.

In [ ]:
print("=== airflow dags test iris_pipeline 2024-01-01 ===")
print("Running all 5 tasks sequentially...\n")

result = subprocess.run(
    [PYTHON, "-m", "airflow", "dags", "test", "iris_pipeline", "2024-01-01"],
    env=env, capture_output=True, text=True
)

# Filter the log stream: keep task stdout, success/failure markers, our prints
keep_keywords = [
    "Loaded ", "Validation passed", "Feature matrix", "Trained on",
    "Accuracy", "precision", "recall", "f1-score", "support",
    "Iris-", "macro avg", "weighted avg",
    "succeeded", "failed", "ERROR", "===",
]
output = result.stdout + result.stderr
for line in output.splitlines():
    if any(kw in line for kw in keep_keywords):
        # Strip the Airflow log prefix [timestamp] {module.py:line}
        if "} INFO -" in line:
            line = line.split("} INFO - ", 1)[-1]
        print(line)

print("\nReturn code:", result.returncode, "(0 = all tasks succeeded)")

## Part 6 — Inspecting XCom values

After a `dags test` run, XCom values are stored in the SQLite metadata DB.
We can query them directly with Python's `sqlite3` module to see exactly what
each task returned and passed downstream.

In [ ]:
import sqlite3, json

db_path = AIRFLOW_HOME / "airflow.db"
conn    = sqlite3.connect(db_path)
cursor  = conn.cursor()

cursor.execute(
    """
    SELECT task_id, key, SUBSTR(value, 1, 120) AS value_preview
    FROM   xcom
    WHERE  dag_id = 'iris_pipeline'
    ORDER  BY timestamp DESC
    """
)
rows = cursor.fetchall()
conn.close()

print(f"{'task_id':<20} {'key':<15} {'value preview':<60}")
print("-" * 95)
for task_id, key, val in rows:
    # val may be JSON-encoded
    try:
        decoded = json.loads(val)
        if isinstance(decoded, list):
            preview = f"[list: {len(decoded)} items]"
        elif isinstance(decoded, dict):
            preview = f"{{dict: {list(decoded.keys())}}}"
        elif isinstance(decoded, str) and len(decoded) > 80:
            preview = decoded[:77] + "..."
        else:
            preview = str(decoded)
    except Exception:
        preview = str(val)[:80]
    print(f"{task_id:<20} {key:<15} {preview:<60}")

**What you see above:**

- `load_data` returned a JSON string of all 150 rows (large — in production,
  return a **file path** or S3 key instead, not the data itself)
- `validate_data` passed it unchanged
- `featurize` returned a dict with `X`, `y`, and `classes` keys
- `train_model` returned `y_test` and `preds` for the evaluation task
- `evaluate` returned the final accuracy score as a float

> **Real-world XCom pattern:** tasks should pass *pointers* (file paths, S3 URIs,
> database table names), not the data itself. The default XCom backend serialises
> values to the metadata DB, which is not designed for large payloads.

## Part 7 — What the Airflow UI shows

In a real deployment you would run `airflow webserver -p 8080` and open
`http://localhost:8080` to see:

| UI view | What you see |
|---------|-------------|
| **DAGs list** | All DAGs with last run status, next scheduled run |
| **Grid view** | A matrix of runs × tasks with colour-coded state (green=success, red=failed, yellow=running) |
| **Graph view** | The DAG topology as a visual graph |
| **Task instance logs** | Full stdout/stderr for each task execution |
| **XCom browser** | All XCom values for a run |

To start the server locally (outside this notebook, in a terminal):

```bash
export AIRFLOW_HOME="$(pwd)/airflow_home"
airflow webserver --port 8080 &
airflow scheduler &
# open http://localhost:8080  (default login: admin / admin)
```

Or use the convenience command that starts both at once:

```bash
airflow standalone
```

## Part 8 — Scheduling and backfill

Change `schedule=None` to a cron expression to make the pipeline run automatically.

In [ ]:
# Demonstrate how cron schedule strings map to intervals
from croniter import croniter
from datetime import datetime as dt

schedules = {
    "Daily at midnight"   : "0 0 * * *",
    "Daily at 06:00"      : "0 6 * * *",
    "Every Monday 09:00"  : "0 9 * * 1",
    "Every 6 hours"       : "0 */6 * * *",
    "1st of every month"  : "0 0 1 * *",
}

base = dt(2024, 6, 1)
print(f"{'Description':<25}  {'Cron':<20}  {'Next 3 runs'}")
print("-" * 85)
for desc, cron in schedules.items():
    try:
        c = croniter(cron, base)
        runs = [str(c.get_next(dt))[:16] for _ in range(3)]
        print(f"{desc:<25}  {cron:<20}  {' | '.join(runs)}")
    except Exception:
        print(f"{desc:<25}  {cron:<20}  (install croniter: pip install croniter)")

In [ ]:
# Backfill: run the DAG for a range of past dates
# Useful for re-processing historical data after fixing a bug.
# With catchup=False (our setting) the scheduler won't do this automatically;
# you trigger it explicitly with `airflow dags backfill`.

print("Command to backfill from 2024-01-01 to 2024-01-03:")
print()
print("  airflow dags backfill iris_pipeline ")
print("    --start-date 2024-01-01")
print("    --end-date   2024-01-03")
print()
print("This would trigger 3 DAG runs (one per day) and run them in dependency order.")
print("We skip actually running this to avoid re-running the training 3× in the notebook.")

## Part 9 — Production patterns

The demo above is intentionally minimal. Here are the patterns you'd add for
a real ML training pipeline:

### 1. Pass file paths, not data
```python
@task
def load_data() -> str:
    path = f"s3://my-bucket/iris/{execution_date}/raw.parquet"
    df = pd.read_csv(DATA_PATH)
    df.to_parquet(path)
    return path          # XCom carries only the path (~50 bytes)
```

### 2. Log metrics to MLflow
```python
@task
def evaluate(results: dict) -> float:
    import mlflow
    acc = accuracy_score(results["y_test"], results["preds"])
    with mlflow.start_run():
        mlflow.log_metric("accuracy", acc)
        mlflow.log_param("model", "LogisticRegression")
    return acc
```

### 3. Alert on failure
```python
@dag(
    ...,
    on_failure_callback=lambda ctx: send_slack_alert(ctx),
)
```

### 4. Retries
```python
@task(retries=3, retry_delay=timedelta(minutes=5))
def load_data():
    ...  # will retry up to 3× if it raises an exception
```

### 5. Sensors (wait for upstream)
```python
from airflow.sensors.s3_key_sensor import S3KeySensor

wait_for_data = S3KeySensor(
    task_id="wait_for_data",
    bucket_name="my-bucket",
    bucket_key="iris/{{ ds }}/raw.csv",
    poke_interval=60,    # check every 60 s
    timeout=3600,        # fail after 1 h
)

wait_for_data >> load_data_task
```

## Key observations

1. **A DAG is just Python.** There is no YAML, no DSL, no special syntax. If it
   imports correctly, Airflow can schedule it.

2. **`@task` + XCom = dataflow.** Returning a value and accepting it as an
   argument is all you need to express dependencies and pass data between tasks.

3. **`dags test` is your development loop.** Write task, run `dags test`, fix,
   repeat. No scheduler or webserver needed during development.

4. **Tasks are isolated processes.** You cannot share in-memory objects between
   tasks. Design your tasks to be stateless and idempotent.

5. **XComs are for metadata, not data.** Pass S3 paths / GCS URIs / database
   table names via XCom. Put the actual bytes in object storage.

6. **Airflow is ops-heavy.** You get a scheduler, webserver, metadata DB, and
   (optionally) distributed workers. Start with `make train` and graduate to
   Airflow only when you need scheduling, retries, or observability at scale.